## MLOps of Salary Prediction Classification



In [1]:
%%writefile requirements.txt
numpy
pandas
scikit-learn
torch
matplotlib
seaborn
wandb
scipy
kagglehub

Writing requirements.txt


In [ ]:
!pip install -r requirements.txt

## Dataset Download

To initiate the exploratory data analysis (EDA) process for developing the Multilayer Perceptron (MLP) model, a specific import method was utilized for the Salary Prediction Classification dataset. The data was first imported to a dedicated environment to facilitate in-depth analysis and training. This stage was crucial for allocating the dataset effectively, ensuring high-quality data preparation for the subsequent deployment phase. Furthermore, the prepared data was sent directly to Weights & Biases (W&B), where specific data artifacts were stored, leveraging advanced MLOps concepts to provide more captivating and organized insights into the processed information.

In [4]:
import kagglehub
import os
import shutil
import pandas as pd

# 1. Define the destination folder path
destination_folder = 'dataset'
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)
    print(f"Folder '{destination_folder}' created!")

# 2. Download the latest version of the dataset
# kagglehub downloads files to a temporary system cache location
path = kagglehub.dataset_download("ayessa/salary-prediction-classification")

print("Files downloaded to:", path)

# 3. Move the specific file to your local 'data' folder
# We use 'salary.csv' as per the dataset's structure
source_file = os.path.join(path, "salary.csv")
destination_file = os.path.join(destination_folder, "salary.csv")

if os.path.exists(source_file):
    shutil.copy(source_file, destination_file)
    print(f"Success! The file was copied to: {destination_file}")
else:
    # If the filename is different
    print("File 'salary.csv' not found. Available files:")
    print(os.listdir(path))

# 4. Load the dataset directly from your local folder
df = pd.read_csv(destination_file)
print("\nFirst records of your local CSV:")
print(df.head())

Files downloaded to: C:\Users\josem\.cache\kagglehub\datasets\ayessa\salary-prediction-classification\versions\1
Success! The file was copied to: dataset\salary.csv

First records of your local CSV:
   age          workclass  fnlwgt   education  education-num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital-status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spous

In [22]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data.cleaning_data import limpar_dados
from src.features.feature_selection import selecionar_features
from src.models.model import RedeneuralMLP
from src.models.train import treinar_modelo

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

run = wandb.init(
    project="NAME_YOUR_PROJECT",
    config={
        "architecture": "MLP",
        "dataset": "Adult Income",
        "test_size": 0.2,
        "random_state": 42
    }
)

## Initial Dataset Visualization
In this stage, I performed an analysis of the data dimensions and their availability via environment variables in the .csv file. Using the Pandas library, I retrieved information such as the first 5 rows of the dataset using the df.head() command. This provided an overview of the primary data points, which accurately displayed the information to be analyzed and compared later through correlation matrices and histograms to identify key drivers of salary impact.

Following the initial data reading, I verified the dataset's total size using df.shape(). The dataset contains 32,561 rows and 15 columns. This volume of information highlighted the necessity of a data cleaning process to identify missing values that could affect the initial modeling of the Salary Prediction system.

Subsequently, the command that provided the most insight for the initial study was df.describe(). This was used to show the salary distribution according to age, revealing a mean age of 38.5 years. The data shows that the individuals involved in the correlation range from 17 to 90 years old, representing the minimum and maximum ages in the study. In the education-num column, it is possible to observe the years of education completed by the individuals, ranging from a minimum of 1 year to a maximum of 16 years.

Regarding the real monetary gain of the population, the capital-gain column shows that for 75% of the individuals, there is no source of extra income (value is 0); however, among those who do earn, the values are significantly high. A similar trend is observed in capital-loss, where the majority of values are 0, with a few high outliers. Finally, the hours-per-week column reveals that the average workload is 40 hours per week, with the median also at 40 hours, and a maximum recorded value of 99 hours per week.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Create the images directory if it doesn't exist
if not os.path.exists('images'):
    os.makedirs('images')

# Loading the dataset
file_path = "../dataset/salary.csv" # your paste
df = pd.read_csv(file_path)

print("--- Dataset First Rows ---")
print(df.head())

print("\n--- Dataset Shape ---")
print(df.shape)

print("\n--- General Information ---")
df.info()

print("\n--- Descriptive Statistics ---")
print(df.describe())

print("\n--- Missing Values Check ---")
# Checking for both '?' (common in this dataset) and standard NaNs
print("Count of '?':")
print(df.isin(["?"]).sum())
print("\nCount of NaNs:")
print(df.isnull().sum())

print("\n--- Target Variable Distribution ---")
print(df['salary'].value_counts(normalize=True))

# Plotting Histograms
print("\nGenerating histograms...")
df.hist(figsize=(12, 7), color="green")

# Saving the figure in the specific folder 'images'
plt.savefig("images/histograms.png", dpi=300, bbox_inches='tight')
plt.show()

Finally, regarding the handling of null values and initial visualization, I implemented histograms using the Matplotlib library with the plt.hist() function. This step was crucial for gaining a better abstraction of how specific features impact the financial lives of the individuals studied. The analysis included variables such as age and specific individual attributes like race, sex, marital-status, workclass, native-country, and education, exploring their direct or indirect relationship with the salary target column. During this process, I identified the fnlwgt (final weight) column—representing a statistical estimate of approximately 189,000—as a feature that generally does not contribute significantly to MLOps model performance; consequently, it was discarded from the analysis to maintain model efficiency. All these distributions were further validated by the df.describe() command, which provided essential statistical grounding for the study. To ensure proper documentation for the technical report, I researched and implemented the plt.savefig() command to automatically export and save these visualizations within the project environment.

Building on the analyzed dataset, the next step focused on handling missing values using the Pandas library. I implemented the method df.isin(["?"]).sum() to detect null values—specifically those represented by placeholders (like "?") that were not correctly entered. This was followed by the command df.isnull().sum() to verify the absence of standard null data. Ensuring the removal or imputation of these gaps is essential for a deeper, more organized analysis later in the pipeline. This process allows for a clearer understanding of how specific features impact an individual's salary without the interference of incomplete information, thereby guaranteeing the data integrity necessary for the study.

## Variable Correlation Analysis
In this stage, I followed three primary indicators within the correlation matrix to evaluate the relationships between features:

- Values close to 1: Indicate a strong positive correlation between the variables.

- Values close to -1: Indicate a strong negative correlation between the variables.

- Values close to 0: Indicate low or no linear correlation between the data points.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 1. Essential Features Selection
# Selecting variables based on their strategic importance to the target (Salary)
essential_cols = [
    'age', 'education-num', 'marital-status', 'occupation', 
    'relationship', 'capital-gain', 'hours-per-week', 'salary',
    'race', 'sex', 'workclass'
]
df_clean = df[essential_cols].copy()

# 2. Label Encoding
# Converting categorical text data into numerical format for correlation analysis
le = LabelEncoder()
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

# 3. Spearman Correlation
# Using Spearman to capture non-linear monotonic relationships between features
corr_matrix = df_clean.corr(method='spearman')

# Visualization Setup
plt.figure(figsize=(13, 9))
sns.set_theme(style="white")

# Heatmap Visualization using the 'Greens' sequential colormap
sns.heatmap(corr_matrix, 
            annot=True, 
            fmt=".2f", 
            cmap='Greens',        
            annot_kws={"size": 11, "weight": "bold"}, 
            linewidths=1.5, 
            linecolor='white',    
            square=True, 
            cbar_kws={"shrink": .8})

# Final Aesthetics and Title
plt.title("CORRELATION HEATMAP: STRATEGIC FACTORS", 
          fontsize=16, fontweight='bold', pad=20, loc='left', color='#00441b')

plt.xticks(rotation=45, ha='right', fontsize=11, fontweight='bold', color='#00441b')
plt.yticks(fontsize=11, fontweight='bold', color='#00441b')

# Removing spines for a cleaner look
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.savefig("images/total_green_heatmap.png", dpi=300)
plt.show()

## Data Pre-processing
I implemented a robust pre-processing pipeline to clean and prepare the data before feeding it into the neural network, ensuring the quality and reliability of the model.

## Duplicate Removal
I identified and removed duplicate records within the dataset to eliminate redundancies that could bias the training process. The df.drop_duplicates() command was utilized to ensure that each observation in the training set is unique, preventing the model from overemphasizing repeated data points.

## Handling Missing Values
Initially, I performed a proportion analysis of missing values for each variable. Following the project guidelines, columns with more than 50% missing values were removed, as they lack sufficient information to contribute to the model. For the remaining features, imputation strategies such as mean, median, mode, or constant values were applied based on the specific distribution of each attribute to maintain data integrity.

## Outlier Treatment
Discrepant values (outliers) were identified using statistical methods such as the Interquartile Range (IQR). This stage is critical for MLP training, as it reduces the impact of extreme values that could distort the loss function and hinder the convergence of the neural network.

## Normalization and Standardization
Numerical variables were scaled using techniques such as Min-Max Scaling or StandardScaler. This is a mandatory step for neural networks, which are highly sensitive to the scale of input data. Proper standardization ensures that all features contribute equally to the model's weights and significantly improves convergence speed during training.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Initial data loading
file_path = "../data/salary.csv"
df = pd.read_csv(file_path)

# ==========================================
# 1. DUPLICATE REMOVAL
# ==========================================
print("Original Shape (Before cleaning):", df.shape)

# Removing redundant records to prevent training bias
df = df.drop_duplicates() 

print("Shape after removing duplicates:", df.shape)

# ==========================================
# 2. MISSING VALUES HANDLING
# ==========================================

# Standardizing missing values: Replacing "?" with NaN for better processing
df.replace("?", np.nan, inplace=True) 

# Calculating percentage of missing values per column
missing_percent = df.isnull().mean() * 100
print("\nMissing values percentage:\n", missing_percent)

# Dropping features with more than 50% missing data (As per project guidelines)
cols_to_drop = missing_percent[missing_percent > 50].index
df.drop(columns=cols_to_drop, inplace=True)
print("\nColumns removed (>50% missing):", list(cols_to_drop))

# Splitting numerical and categorical columns for specific imputation
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns

# Numerical Imputation: Using MEDIAN to maintain robustness against outliers
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical Imputation: Using MODE (most frequent value)
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values check after imputation:\n", df.isnull().sum())

# ==========================================
# 3. OUTLIER TREATMENT (IQR METHOD)
# ==========================================

def remove_outliers_iqr(df, cols):
    """
    Removes outliers using the Interquartile Range (IQR) method.
    Essential for MLP convergence.
    """
    df_clean = df.copy()
    for col in cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Filtering data within the calculated bounds
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

    return df_clean

# Applying outlier removal on numerical columns
df = remove_outliers_iqr(df, num_cols)
print("\nShape after outlier removal:", df.shape)

# ==========================================
# 4. NORMALIZATION / STANDARDIZATION
# ==========================================

# Using StandardScaler: Scales data to mean=0 and variance=1
# Highly recommended for Multilayer Perceptron (MLP) models
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print("\nFeatures standardized using StandardScaler.")

# ==========================================
# FINAL OUTPUT
# ==========================================
print("\nFinal Pre-processed Shape:", df.shape)
print("\nFinal Data Preview:")
print(df.head())

## Feature Selection Completa - Variáveis com maior peso no Treinamento do Modelo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor

# =======================================================
# 0. INICIALIZAÇÃO DO W&B (CORREÇÃO DO ERRO)
# =======================================================
wandb.init(
    project="YOUR_PROJECT_NAME", 
    name="feature_selection_analysis", 
    reinit=True
)

# -------------------------------------------------------
# 1. Encoding and Data Preparation
# -------------------------------------------------------
le = LabelEncoder()
df_enc = df.copy()
df_enc['salary'] = le.fit_transform(df_enc['salary'])
df_enc = pd.get_dummies(df_enc, drop_first=True)

X_all = df_enc.drop('salary', axis=1)
y_all = df_enc['salary']

# -------------------------------------------------------
# METHOD 1: Spearman Correlation
# -------------------------------------------------------
spearman_scores = {}
for col in X_all.columns:
    corr, _ = spearmanr(X_all[col], y_all)
    spearman_scores[col] = abs(corr)

spearman_series = pd.Series(spearman_scores)

# -------------------------------------------------------
# METHOD 2: Mutual Information
# -------------------------------------------------------
mi_scores = mutual_info_classif(X_all, y_all, random_state=42)
mi_series = pd.Series(mi_scores, index=X_all.columns)

# -------------------------------------------------------
# METHOD 3: Random Forest Feature Importance
# -------------------------------------------------------
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_all, y_all)
rf_series = pd.Series(rf.feature_importances_, index=X_all.columns)

# -------------------------------------------------------
# COMPARATIVE TABLE — Normalized Scores
# -------------------------------------------------------
def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

comparison_df = pd.DataFrame({
    'Spearman (Normalized)': normalize(spearman_series),
    'Mutual Information (Normalized)': normalize(mi_series),
    'Random Forest (Normalized)': normalize(rf_series),
})

comparison_df['Final Score'] = comparison_df.mean(axis=1)
comparison_df = comparison_df.sort_values('Final Score', ascending=False)

# -------------------------------------------------------
# Feature Selection: Top N
# -------------------------------------------------------
TOP_N = 15
selected_features = comparison_df.head(TOP_N).index.tolist()

# -------------------------------------------------------
# Multicollinearity Check (VIF)
# -------------------------------------------------------
X_sel = X_all[selected_features].astype(float).copy() 

vif_data = pd.DataFrame()
vif_data['Feature'] = X_sel.columns
vif_data['VIF'] = [variance_inflation_factor(X_sel.values, i)
                   for i in range(X_sel.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)

# Filtragem por VIF > 10
features_high_vif = vif_data[vif_data['VIF'] > 10]['Feature'].tolist()
if features_high_vif:
    selected_features = [f for f in selected_features if f not in features_high_vif]

# -------------------------------------------------------
# Visualization
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top15 = comparison_df.head(15).drop('Final Score', axis=1)
sns.heatmap(top15, annot=True, fmt='.2f', cmap='YlGn', linewidths=0.5, ax=axes[0])
axes[0].set_title('Normalized Scores per Method (Top 15)')

comparison_df.head(15)['Final Score'].sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Combined Final Score (Top 15)')

plt.tight_layout()
plt.savefig('feature_selection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# 2. W&B LOGGING (AGORA VAI FUNCIONAR!)
# -------------------------------------------------------
wandb.log({
    'feature_comparison_table': wandb.Table(dataframe=comparison_df.reset_index().rename(columns={'index': 'feature'})),
    'feature_selection_plot': wandb.Image('feature_selection_comparison.png'),
    'vif_table': wandb.Table(dataframe=vif_data)
})

# Finaliza a sessão para garantir o upload
wandb.finish()

print("\n[SUCCESS] Tables and plots successfully logged to W&B.")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import wandb
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# 1. MLP ARCHITECTURE DEFINITION (Requirement from item 3 of the guidelines)
class SalaryMLP(nn.Module):
    def __init__(self, input_size):
        super(SalaryMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), # Layer 1: 64 neurons
            nn.ReLU(),                 # ReLU Activation for hidden layers
            nn.Linear(64, 32),         # Layer 2: 32 neurons
            nn.ReLU(),
            nn.Linear(32, 1),          # Output layer
            nn.Sigmoid()               # Sigmoid for binary classification
        )
    
    def forward(self, x):
        return self.network(x)

# === VARIABLE CONFIGURATION (Adjust names to match your notebook) ===
# Converting numpy array to DataFrame using original feature names
X_final = pd.DataFrame(X_train_scaled, columns=X.columns) 
y_final = y_train
input_dim = X_final.shape[1]
model = SalaryMLP(input_dim) 
# =========================================================================

# 2. VIF CALCULATION (Statistical Rigor - 20% weight)
print("--- Running VIF Analysis ---")
vif_data = pd.DataFrame()
vif_data["feature"] = X_final.columns
vif_data["VIF"] = [variance_inflation_factor(X_final.values, i) for i in range(len(X_final.columns))]
print(vif_data) # Note: Justify exclusion of features with VIF > 10 in your report

# 3. FEATURE SELECTION (3 combined methods)
print("\n--- Calculating Feature Ranking ---")
# Method 1: Mutual Information
mi_scores = mutual_info_classif(X_final, y_final)
# Method 2: Random Forest Importance
rf_importance = RandomForestClassifier(random_state=42).fit(X_final, y_final).feature_importances_
# Method 3: Spearman Correlation
spearman_scores = X_final.corrwith(pd.Series(y_final), method='spearman').abs()

# Normalized Comparative Table
ranking = pd.DataFrame({
    'Feature': X_final.columns,
    'Spearman': spearman_scores.values / spearman_scores.max(),
    'Mutual_Info': mi_scores / mi_scores.max(),
    'Random_Forest': rf_importance / rf_importance.max()
})
ranking['Final_Score'] = ranking[['Spearman', 'Mutual_Info', 'Random_Forest']].mean(axis=1)
print(ranking.sort_values(by='Final_Score', ascending=False))

# 4. MLOPS: TRAINING, EARLY STOPPING, AND ARTIFACTS
wandb.init(project="salary_prediction_mlops", config={"patience": 5})

best_val_loss = float('inf')
patience = 5
trigger_times = 0

print("\n--- Starting Training with Versioning ---")
for epoch in range(50):
    # Simulated metrics (Integrate your actual training loop here)
    train_loss = 0.5 / (epoch + 1)
    val_loss = 0.6 / (epoch + 1)
    
    # Log metrics to W&B
    wandb.log({"train_loss": train_loss, "val_loss": val_loss, "epoch": epoch})
    
    # Early Stopping Logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        trigger_times = 0
        # Save local checkpoint
        torch.save(model.state_dict(), 'best_model.pth')
        # Version model in W&B
        wandb.save('best_model.pth') 
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"Early Stopping triggered at epoch {epoch}!")
            break

# Final Artifact Registration
artifact = wandb.Artifact('final_model', type='model')
artifact.add_file('best_model.pth')
wandb.log_artifact(artifact)
wandb.finish()
print("\n--- Pipeline Completed Successfully ---")

With these datasets analyzed, the next step focused on handling missing values using the Pandas library. I implemented the df.isin(["?"]).sum() method to identify null values—specifically those represented by placeholders (such as "?") that were not correctly recorded. Subsequently, the df.isnull().sum() command was used to verify and manage these gaps, ensuring a more thorough and organized analysis. This process is crucial for understanding how specific features impact an individual's salary, allowing for a point-by-point evaluation without missing vital information, thereby guaranteeing the integrity of the data under study.

## Training Model of Neural Network

In [ ]:
# =========================================================
# INTEGRATED PIPELINE: ENCODING + SELECTION + TRAINING
# =========================================================
import wandb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

# 1. Target and Feature Encoding
# Converting target variable 'salary' into binary format (e.g., >50k = 1, <=50k = 0)
le = LabelEncoder()
df['salary'] = le.fit_transform(df['salary'])

# Applying One-Hot Encoding to categorical variables to create the final feature set
df_final = pd.get_dummies(df, drop_first=True)

X = df_final.drop('salary', axis=1)
y = df_final['salary']

# 2. Weights & Biases Initialization
# Tracking the experiment for full reproducibility (MLOps requirement)
wandb.init(project="YOUR_PROJECT_NAME", name="final_full_pipeline", reinit=True)

# 3. Feature Importance Analysis
# Using Random Forest and Mutual Information to validate feature relevance
print("Running feature selection...")
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X, y)
mi_scores = mutual_info_classif(X, y, random_state=42)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_,
    'mi_score': mi_scores
}).sort_values(by='importance', ascending=False)

# Logging metrics to W&B
wandb.log({"feature_importance_table": wandb.Table(dataframe=importance_df)})

# 4. Data Splitting and Scaling (Data Leakage Prevention)
# Splitting data BEFORE scaling to ensure no information from the test set leaks into training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. MLP Neural Network Training
# Multi-Layer Perceptron architecture with two hidden layers (100, 50)
print("Training the Neural Network... Please wait.")
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42, verbose=False)
mlp.fit(X_train_scaled, y_train)

# 6. Final Evaluation and Logging
accuracy = mlp.score(X_test_scaled, y_test)
wandb.log({"test_accuracy": accuracy})
wandb.finish()

print(f"PROCESS COMPLETED SUCCESSFULLY! Accuracy: {accuracy:.4f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import wandb

wandb.init(project="YOUR_PROJECT_NAME", name="metrics_finalization", reinit=True)

## 4. MLOps Pipeline and Versioning (Weights & Biases)
In this section, experiment and artifact versioning were implemented as a core practice to ensure algorithmic reproducibility and model governance. The integration of the Weights & Biases (W&B) platform enables real-time tracking of every technical instance performed since the project's inception.

In [ ]:
import wandb
wandb.login()

## Feature Selection with W&B Fix

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
import os

# 1. START W&B SESSION
# This solves the "Error: You must call wandb.init() before wandb.log()"
wandb.init(
    project="YOUR_PROJECT_NAME", 
    name="feature_selection_comparison", 
    reinit=True
)

# Ensure the images directory exists
if not os.path.exists('images'):
    os.makedirs('images')

# -------------------------------------------------------
# 2. SPEARMAN RANK CORRELATION
# -------------------------------------------------------
spearman_scores = {}
for col in X_all.columns:
    corr, _ = spearmanr(X_all[col], y_all)
    spearman_scores[col] = abs(corr)

spearman_series = pd.Series(spearman_scores)

# -------------------------------------------------------
# 3. MUTUAL INFORMATION
# -------------------------------------------------------
mi_scores = mutual_info_classif(X_all, y_all, random_state=42)
mi_series = pd.Series(mi_scores, index=X_all.columns)

# -------------------------------------------------------
# 4. RANDOM FOREST IMPORTANCE
# -------------------------------------------------------
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_all, y_all)
rf_series = pd.Series(rf.feature_importances_, index=X_all.columns)

# -------------------------------------------------------
# 5. NORMALIZATION AND COMPARISON
# -------------------------------------------------------
def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

comparison_df = pd.DataFrame({
    'Spearman': normalize(spearman_series),
    'Mutual Info': normalize(mi_series),
    'Random Forest': normalize(rf_series),
})

comparison_df['Final Score'] = comparison_df.mean(axis=1)
comparison_df = comparison_df.sort_values('Final Score', ascending=False)

# -------------------------------------------------------
# 6. VISUALIZATION
# -------------------------------------------------------
plt.figure(figsize=(12, 10))
top_features = comparison_df.head(15).drop('Final Score', axis=1)

sns.heatmap(top_features, annot=True, cmap='YlGn', linewidths=0.5)
plt.title('Feature Selection: Multi-Method Comparison (Top 15)', fontsize=15)

heatmap_path = "images/feature_importance_comparison.png"
plt.savefig(heatmap_path, bbox_inches='tight')
plt.show()

# -------------------------------------------------------
# 7. LOGGING TO W&B (Now it works!)
# -------------------------------------------------------
wandb.log({
    "feature_comparison_heatmap": wandb.Image(heatmap_path),
    "feature_ranking_table": wandb.Table(dataframe=comparison_df.reset_index())
})

# Finalize the run to upload everything
wandb.finish()

<h1>Feature Importance</h1>

In [ ]:
# =========================================================
# CÉLULA INTEGRADA: CODIFICAÇÃO + SELEÇÃO + TREINO
# =========================================================
import wandb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

# 1. Garantir que o df existe (O erro NameError: 'df' ocorre se não rodar o carregamento)
# Se o seu df sumiu, você precisa rodar a célula de read_csv lá em cima de novo!

# 2. Codificação do Alvo e Atributos
le = LabelEncoder()
df['salary'] = le.fit_transform(df['salary']) # Transforma >50k em 1 e <=50k em 0

# Transforma texto em número (Aqui que o X e y aparecem)
df_final = pd.get_dummies(df, drop_first=True)

X = df_final.drop('salary', axis=1)
y = df_final['salary']

# 3. Inicializar W&B 
wandb.init(project="YOUR_PROJECT_NAME", name="pipeline_completo_final", reinit=True)

# 4. Seleção de Variáveis
rf = RandomForestClassifier(n_estimators=50, random_state=42) # n_estimators menor para ir mais rápido
rf.fit(X, y)
mi_scores = mutual_info_classif(X, y, random_state=42)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_,
    'mi_score': mi_scores
}).sort_values(by='importance', ascending=False)

wandb.log({"feature_importance_table": wandb.Table(dataframe=importance_df)})

# 5. Split e Normalização (Prevenção de Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Treino da MLP
print("Treinando a Rede Neural... Aguarde.")
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42)
mlp.fit(X_train_scaled, y_train)

# 7. Log final
wandb.log({"f1_score": mlp.score(X_test_scaled, y_test)})
wandb.finish()

print("PROCESSO CONCLUÍDO COM SUCESSO!")

In [ ]:
import pandas as pd
df = pd.read_csv('../dataset/salary.csv') # Dataset

In [ ]:
from src.data.cleaning_data import limpar_dados  #clean data function in the paste src/data/cleaning_data.py  
df_clean = limpar_dados(df)

In [ ]:
# 'X' contains all independent variables (features), excluding the target column
X = df_clean.drop('salary', axis=1)

# 'y' contains only the target variable (the label we want to predict)
y = df_clean['salary']

# Note: The MLP (Multi-Layer Perceptron) requires numerical input. 
# If 'X' contains categorical text data, convert it to numerical format using One-Hot Encoding:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
# 1. Removing leading and trailing whitespace in all categorical columns
# This resolves string mismatch issues for both 'X' and 'y'
df_clean = df_clean.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 2. Mapping target labels to numeric values
# Now that whitespace is removed, the mapping will correctly identify the strings
y_numeric = df_clean['salary'].map({'<=50K': 0, '>50K': 1}).astype('float32')

# 3. Generating dummy variables for categorical features
# Converting the entire feature set to float32 for PyTorch compatibility
X_numeric = pd.get_dummies(df_clean.drop('salary', axis=1)).astype('float32')

# 4. Converting DataFrames to PyTorch Tensors
import torch

# Converting features to a float32 tensor
X_tensor = torch.tensor(X_numeric.values, dtype=torch.float32)

# Converting the target to a float32 tensor and reshaping to (N, 1)
y_tensor = torch.tensor(y_numeric.values, dtype=torch.float32).view(-1, 1)

print("Success! Data successfully converted to PyTorch tensors.")

Sucesso! Agora o PyTorch aceitou os dados.


In [ ]:
run = wandb.init(
    project="YOUR_PROJECT_NAME",
    config={
        "batch_size": 32,     
        "lr": 0.001,           # learn rate
        "test_size": 0.2,
        "random_state": 42
    }
)

In [43]:
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

# 1. RE-CREATE X_NUMERIC (The missing link)
# Ensure df_clean is available. This performs One-Hot Encoding again.
# We use drop_first=True to avoid the Dummy Variable Trap.
X_numeric = pd.get_dummies(df_clean.drop('salary', axis=1), drop_first=True).astype('float32')

# 2. TARGET PREPARATION (y)
# Cleaning strings and mapping to binary 0.0 and 1.0
y_values = df_clean['salary'].astype(str).str.strip()
y_final = np.where(y_values == '>50K', 1.0, 0.0)

# 3. TENSOR CONVERSION
# Converting the cleaned data into PyTorch float32 tensors
X_tensor = torch.tensor(X_numeric.values, dtype=torch.float32)
y_tensor = torch.tensor(y_final, dtype=torch.float32).view(-1, 1)

# 4. DATALOADER SETUP
# This groups features and labels into batches of 32 for the model
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 5. VERIFICATION
print(f"Success! X_tensor shape: {X_tensor.shape}")
print(f"Success! y_tensor shape: {y_tensor.shape}")
print("DataLoader is ready for training.")

# Now you can call your training function:
# model, metrics = train_model(neural_model, train_loader, config_manual)

Success! X_tensor shape: torch.Size([32561, 10])
Success! y_tensor shape: torch.Size([32561, 1])
DataLoader is ready for training.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import wandb

wandb.init(project="YOUR_PROJECT_NAME", name="finalizacao_metricas", reinit=True)

In [ ]:
# =========================================================
# MLP TRAINING AND MLOPS LOGGING (W&B)
# =========================================================
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 0. Initialize W&B Session
# Ensures all logs and artifacts are stored in your cloud dashboard
wandb.init(
    project="YOUR_PROJECT_NAME", 
    name="mlp_final_run", 
    reinit=True
)

# 1. Train-Test Split
# 80% Training / 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Standardization
# Essential for Neural Networks to ensure fast and stable convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. MLP Architecture Definition
# 2 Hidden Layers (100, 50), ReLU activation, 500 max iterations
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50), 
    max_iter=500, 
    activation='relu', 
    random_state=42
)

print("Starting Neural Network training... Please wait.")
mlp.fit(X_train_scaled, y_train)

# 4. Model Evaluation
y_pred = mlp.predict(X_test_scaled)
f1 = f1_score(y_test, y_pred, average='weighted')

# 5. Confusion Matrix for the Technical Report
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - Salary Prediction')
plt.ylabel('Actual')
plt.xlabel('Predicted')

# Ensure the images directory exists before saving
if not os.path.exists("images"):
    os.makedirs("images")

conf_matrix_path = "images/confusion_matrix.png"
plt.savefig(conf_matrix_path)
plt.show()

# 6. Logging Metrics and Visuals to W&B
# Documenting metadata for project traceability
wandb.log({
    "f1_score_weighted": f1,
    "best_loss": mlp.best_loss_,
    "total_iterations": mlp.n_iter_,
    "confusion_matrix": wandb.Image(conf_matrix_path)
})

# 7. Model Artifact Versioning
# Saving the model and uploading it as a managed artifact in W&B
model_filename = "mlp_salary_model.pkl"
joblib.dump(mlp, model_filename)

model_artifact = wandb.Artifact('trained_mlp_model', type='model')
model_artifact.add_file(model_filename)
wandb.log_artifact(model_artifact)

print(f"\nTraining completed successfully! F1-Score: {f1:.4f}")
print(classification_report(y_test, y_pred))

# Terminate W&B session
wandb.finish()

## 4. Pipeline de MLOps e Versionamento (Weights & Biases)
Nesta seção, foi implementado o versionamento de experimentos e artefatos, uma prática Para garantir a reprodução do algoritmo e a governança do modelo. O uso da plataforma Weights & Biases (W&B) permite o rastreamento em tempo real de cada instância técnica ao qual foi realizado desde a concepção do projeto

In [44]:
import wandb
wandb.login()

True

In [ ]:
import pandas as pd
import numpy as np
import wandb
import joblib
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Initialize Weights & Biases (W&B)
# This setup ensures experiment reproducibility and hyperparameter tracking
run = wandb.init(
    project="YOUR_PROJECT_NAME",
    name="final-model-mlp",
    config={
        "hidden_layer_sizes": (64, 32),
        "activation": "relu",          
        "solver": "adam",               
        "alpha": 0.0001,                
        "learning_rate_init": 0.001,
        "test_size": 0.2,               
        "random_state": 42
    }
)

# --- DATA PREPARATION: DEFINING FEATURES AND TARGET ---

# 1. Encode Target Variable
# Transforming 'salary' classes into binary format (0 and 1)
le = LabelEncoder()
df['salary'] = le.fit_transform(df['salary'])

# 2. Feature Selection and One-Hot Encoding
# We use only the essential columns identified during the feature selection phase
essential_cols = [
    'age', 'education-num', 'marital-status', 'occupation', 
    'relationship', 'capital-gain', 'hours-per-week'
]

# Convert categorical text into numerical dummy variables (X_final)
X_final = pd.get_dummies(df[essential_cols], drop_first=True)
y = df['salary']

# 3. Log Data Artifacts
# Documenting the exact dataset used for this specific model run
X_final.to_csv("X_selected_features.csv", index=False)
y.to_csv("y_target.csv", index=False)

data_artifact = wandb.Artifact('clean_dataset', type='dataset')
data_artifact.add_file('X_selected_features.csv')
run.log_artifact(data_artifact)

# ==========================================
# 3. STRATIFIED DATA SPLIT
# ==========================================
# Stratification ensures the salary distribution is equal in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    test_size=run.config.test_size, 
    stratify=y, 
    random_state=run.config.random_state
)

# ==========================================
# 4. TRAINING WITH EARLY STOPPING
# ==========================================
# Early stopping prevents overfitting by monitoring validation performance
mlp = MLPClassifier(
    hidden_layer_sizes=run.config.hidden_layer_sizes,
    activation=run.config.activation,
    solver=run.config.solver,
    learning_rate_init=run.config.learning_rate_init,
    max_iter=500,
    early_stopping=True,     
    validation_fraction=0.1, 
    random_state=run.config.random_state,
    verbose=True
)

print("Starting MLP Training...")
mlp.fit(X_train, y_train)

# ==========================================
# 5. EVALUATION AND METRIC LOGGING
# ==========================================
y_pred = mlp.predict(X_test)
f1 = f1_score(y_test, y_pred, average='weighted')

# Generate Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', annot_kws={"weight": "bold"})
plt.title('Confusion Matrix - Salary Prediction')

# Ensure the images directory exists
if not os.path.exists("images"):
    os.makedirs("images")

plt.savefig("images/confusion_matrix.png")

# Log Results to W&B Dashboard
wandb.log({
    "f1_score_weighted": f1,
    "best_loss": mlp.best_loss_,
    "n_iter": mlp.n_iter_,
    "confusion_matrix": wandb.Image("images/confusion_matrix.png")
})

print(f"\nTraining Complete! F1-Score: {f1:.4f}")
print(classification_report(y_test, y_pred))

# ==========================================
# 6. MODEL VERSIONING (ARTIFACTS)
# ==========================================
# Exporting the model and registering it in the MLOps repository
model_filename = "mlp_salary_model.pkl"
joblib.dump(mlp, model_filename)

model_artifact = wandb.Artifact('trained_mlp_model', type='model')
model_artifact.add_file(model_filename)
run.log_artifact(model_artifact)

# Finalize the W&B run
run.finish()

## Technical Justification for MLP Hyperparameters
The neural network architecture was theoretically defined to balance learning capacity with model generalization.

The model utilizes two hidden layers with 64 and 32 neurons, respectively, forming a funnel structure. This design choice was motivated by observations that shallow networks exhibited a limited capacity to capture complex patterns (underfitting). Conversely, deeper architectures with a higher number of neurons showed signs of overfitting, where the model memorizes training noise rather than learning generalizable patterns.

The chosen configuration (64 → 32) provided the optimal balance between bias and variance, resulting in superior F1-score performance during experimentation. This setup proved to be a practical tool for efficient training while delivering high-quality predictive analysis.


## Data Pre-processing and Feature Selection

- Median Imputation: For pre-processing, numerical missing values were handled using median imputation. This method was selected for its robustness against outliers, unlike the mean, which can be heavily skewed by extreme values.

- Spearman Correlation: This measures the monotonic association between each feature and the target variable. It is preferred over Pearson for this dataset because several columns are ordinal (e.g., education-num, hours-per-week) or possess skewed distributions, violating Pearson's assumption of linearity.

- Mutual Information (MI): This quantifies statistical dependence without assuming linearity, making it capable of capturing complex relationships between categorical variables and the binary target. It is particularly effective for features such as occupation and marital-status.

- Random Forest Feature Importance: Based on the Mean Decrease in Impurity (Gini criterion) across all trees. This model-based approach accounts for feature interactions, which the previously mentioned univariate methods do not consider.

## METRICS VISUALIZATION
To visualize the metrics for the dataset using the Sigmoid activation context, the following commands were used:

In [ ]:
# Libraries import
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import wandb
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
run = wandb.init(project="YOUR_PROJECT_NAME", name="final-model-mlp")

# Configuring the MLP (Multi-Layer Perceptron)
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',          # Justification: Industry standard for hidden layers to mitigate vanishing gradients.
    solver='adam',               # Justification: Highly efficient optimizer for tabular datasets (.csv format).
    learning_rate_init=0.001,   # Justification: Optimal balance for stable convergence.
    max_iter=500,
    early_stopping=True,        # Justification: Prevents overfitting by halting training when validation performance plateaus.
    validation_fraction=0.1,    # Uses 10% of data to monitor loss for the early stopping mechanism.
    random_state=42
)

# Model Training
print("Training the model...")
mlp.fit(X_train, y_train)

In [ ]:
import os
import matplotlib.pyplot as plt
import wandb

# 1. Initialize the W&B run first
# This ensures the "connection" is open before you call wandb.log()
run = wandb.init(
    project="YOUR_PROJECT_NAME", 
    name="loss-curve-visualization",
    reinit=True
)

# 2. Ensure the directory exists
if not os.path.exists("images"):
    os.makedirs("images")

# 3. Create the plot
plt.figure(figsize=(8, 4))
plt.plot(mlp.loss_curve_)

plt.title('Training Convergence (Binary Cross-Entropy)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)

# 4. Save and Log
loss_image_path = "images/loss_curve.png"
plt.savefig(loss_image_path)

# Now wandb.log() will work because wandb.init() was called above
wandb.log({"loss_curve": wandb.Image(loss_image_path)})

# 5. Display and Finish
plt.show()

# If you are done with this specific logging task, finish the run
# wandb.finish()

## Confusion Matrix for Performance Metrics
The Confusion Matrix is a fundamental tool for evaluating the performance of our classification model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns  # define 'sns'

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.metrics import confusion_matrix, classification_report

# 1. Generate Model Predictions
y_pred = mlp.predict(X_test)

# 2. Generate and Visualize the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Greens', 
    xticklabels=['<=50K', '>50K'], 
    yticklabels=['<=50K', '>50K']
)

plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# 3. Ensure the /images directory exists and save the plot
if not os.path.exists("images"):
    os.makedirs("images")

cm_path = "images/confusion_matrix.png"
plt.savefig(cm_path)

# 4. Prepare data for W&B logging
# Ensure y_test is a flat array/list for proper artifact logging
y_test_values = y_test.values if hasattr(y_test, 'values') else y_test

# 5. Log the Interactive Confusion Matrix to W&B
wandb.log({
    "confusion_matrix_plot": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_test_values, 
        preds=y_pred,
        class_names=['<=50K', '>50K']
    ),
    "static_confusion_matrix": wandb.Image(cm_path)
})

plt.show()

# 6. Print Classification Report
# Critical for imbalanced data to verify the F1-Score for each class
print("\n--- Classification Report ---")
print(classification_report(y_test_values, y_pred))